In [ ]:
# Import core libraries and align the working directory with the repo root.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

# Resolve project root so `src` is importable in notebooks.
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
os.chdir(project_root)

from src.tune import load_from_csv
from src.metrics import evaluate_model_performance

tabular_path = project_root / "data/transform/online_retail_daily_product_tabular.csv"
X_train, y_train, X_test, y_test, avg_price_arr = load_from_csv(str(tabular_path))

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)


In [ ]:
# Stage 1: classify demand presence to address zero inflation early.
y_train_zero = (y_train > 0).astype(int)
count_zero = int((y_train_zero == 0).sum())
count_one = int((y_train_zero == 1).sum())
imbalance_ratio = count_zero / count_one if count_one > 0 else 1.0

clf = xgb.XGBClassifier(scale_pos_weight=imbalance_ratio)
clf.fit(X_train, y_train_zero)


In [ ]:
# Stage 2: regress demand magnitude only on non-zero samples.
nonzero_mask = y_train > 0
X_train_nonzero = X_train.loc[nonzero_mask]
y_train_nonzero = y_train.loc[nonzero_mask]

X_train_nonzero = X_train_nonzero.astype(np.float32)

reg = xgb.XGBRegressor()
reg.fit(X_train_nonzero, y_train_nonzero)


In [ ]:
# Combine classification thresholding with regression output for final forecast.
proba_nonzero = clf.predict_proba(X_test)[:, 1]
BUSINESS_THRESHOLD = 0.30
zero_pred_adjusted = (proba_nonzero >= BUSINESS_THRESHOLD).astype(int)

qty_pred = reg.predict(X_test)
final_pred = zero_pred_adjusted * qty_pred


In [ ]:
# Evaluate using the centralized metric suite for statistical and business alignment.
metrics = evaluate_model_performance(
    y_true=y_test.values if hasattr(y_test, "values") else y_test,
    y_pred=final_pred,
    unit_prices=avg_price_arr,
    gross_margin=0.20,
    cogs=0.80,
    holding_daily=0.20 / 365,
)

metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["value"])
metrics_df


In [ ]:
# Visualize forecasts for highest-demand products without loading full data into memory.
import matplotlib.pyplot as plt

use_cols = ["stock_code", "date", "demand_qty"]
max_date = None
for chunk in pd.read_csv(tabular_path, usecols=use_cols, chunksize=200_000, low_memory=False):
    chunk["date"] = pd.to_datetime(chunk["date"])
    chunk_max = chunk["date"].max()
    max_date = chunk_max if max_date is None else max(max_date, chunk_max)

cutoff = max_date - pd.Timedelta(days=30)

test_chunks = []
for chunk in pd.read_csv(tabular_path, usecols=use_cols, chunksize=200_000, low_memory=False):
    chunk["date"] = pd.to_datetime(chunk["date"])
    test_chunk = chunk[chunk["date"] > cutoff]
    if not test_chunk.empty:
        test_chunks.append(test_chunk)

df_test = pd.concat(test_chunks, ignore_index=True)
df_test = df_test.sort_values(["stock_code", "date"]).reset_index(drop=True)

if len(df_test) != len(y_test):
    raise ValueError("Mismatch between test rows and predictions.")

df_test["y_true"] = y_test.values
df_test["y_pred"] = final_pred

top_k = 5
top_products = (
    df_test.groupby("stock_code")["y_true"]
    .sum()
    .sort_values(ascending=False)
    .head(top_k)
    .index
)

fig, axes = plt.subplots(top_k, 1, figsize=(12, 3 * top_k), sharex=True)
if top_k == 1:
    axes = [axes]

for ax, stock_code in zip(axes, top_products):
    subset = df_test[df_test["stock_code"] == stock_code]
    ax.plot(subset["date"], subset["y_true"], label="Actual", linewidth=1.5)
    ax.plot(subset["date"], subset["y_pred"], label="Forecast", linewidth=1.5)
    ax.set_title(f"Stock {stock_code} - Top Demand")
    ax.set_ylabel("Demand")
    ax.legend()

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
# Ensure optional boosting libraries are available before training model zoo.
import importlib
import subprocess
import sys

missing = []
for pkg in ["lightgbm", "catboost"]:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All optional boosting dependencies already installed.")


In [ ]:
# Import model zoo dependencies after ensuring installation.
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import precision_score, recall_score, roc_curve, auc
from sklearn.metrics import mean_absolute_error, mean_squared_error
import gc


In [ ]:
# Train and evaluate all classifier-regressor boosting combinations with memory guards.
y_train_zero = (y_train > 0).astype(int).values
y_test_zero = (y_test > 0).astype(int).values

count_zero = int((y_train_zero == 0).sum())
count_one = int((y_train_zero == 1).sum())
imbalance_ratio = count_zero / count_one if count_one > 0 else 1.0

classifier_factories = {
    "xgboost": lambda: xgb.XGBClassifier(scale_pos_weight=imbalance_ratio),
    "lightgbm": lambda: lgb.LGBMClassifier(scale_pos_weight=imbalance_ratio),
    "catboost": lambda: CatBoostClassifier(verbose=False, scale_pos_weight=imbalance_ratio),
    "adaboost": lambda: AdaBoostClassifier(),
    "gradient_boost": lambda: GradientBoostingClassifier(),
}

regressor_factories = {
    "xgboost": lambda: xgb.XGBRegressor(),
    "lightgbm": lambda: lgb.LGBMRegressor(),
    "catboost": lambda: CatBoostRegressor(verbose=False),
    "adaboost": lambda: AdaBoostRegressor(),
    "gradient_boost": lambda: GradientBoostingRegressor(),
}

nonzero_mask = y_train > 0
X_train_nonzero = X_train.loc[nonzero_mask]
y_train_nonzero = y_train.loc[nonzero_mask]

combination_results = []
classifier_eval_rows = []
regressor_eval_rows = []
classifier_predictions = {}
regressor_predictions = {}
combined_predictions = {}

BUSINESS_THRESHOLD = 0.30

for clf_name, clf_factory in classifier_factories.items():
    clf = clf_factory()
    clf.fit(X_train, y_train_zero)

    proba_nonzero = clf.predict_proba(X_test)[:, 1]
    zero_pred = (proba_nonzero >= BUSINESS_THRESHOLD).astype(int)

    precision = precision_score(y_test_zero, zero_pred, zero_division=0)
    recall = recall_score(y_test_zero, zero_pred, zero_division=0)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    fpr, tpr, _ = roc_curve(y_test_zero, proba_nonzero)
    roc_auc = auc(fpr, tpr)

    classifier_eval_rows.append({
        "classifier": clf_name,
        "F1_Zero": f1,
        "Precision": precision,
        "Recall": recall,
        "ROC_AUC": roc_auc,
    })

    classifier_predictions[clf_name] = {
        "proba": proba_nonzero,
        "binary": zero_pred,
    }

    for reg_name, reg_factory in regressor_factories.items():
        reg = reg_factory()
        reg.fit(X_train_nonzero, y_train_nonzero)
        qty_pred = reg.predict(X_test)

        nonzero_test_mask = y_test.values > 0
        rmse = np.sqrt(mean_squared_error(y_test.values[nonzero_test_mask], qty_pred[nonzero_test_mask])) if nonzero_test_mask.any() else 0.0
        mae = mean_absolute_error(y_test.values[nonzero_test_mask], qty_pred[nonzero_test_mask]) if nonzero_test_mask.any() else 0.0
        wape = (np.abs(y_test.values[nonzero_test_mask] - qty_pred[nonzero_test_mask]).sum() / y_test.values[nonzero_test_mask].sum()) if nonzero_test_mask.any() else 0.0
        bias = ((qty_pred[nonzero_test_mask].sum() - y_test.values[nonzero_test_mask].sum()) / y_test.values[nonzero_test_mask].sum()) if nonzero_test_mask.any() else 0.0

        regressor_eval_rows.append({
            "regressor": reg_name,
            "MAE_NonZero": mae,
            "RMSE_NonZero": rmse,
            "WAPE_NonZero": wape,
            "Bias_NonZero": bias,
        })

        regressor_predictions.setdefault(reg_name, qty_pred)

        final_pred_combo = zero_pred * qty_pred
        combined_predictions[f"{clf_name}__{reg_name}"] = final_pred_combo

        metrics = evaluate_model_performance(
            y_true=y_test.values if hasattr(y_test, "values") else y_test,
            y_pred=final_pred_combo,
            unit_prices=avg_price_arr,
            gross_margin=0.20,
            cogs=0.80,
            holding_daily=0.20 / 365,
        )
        row = {
            "classifier": clf_name,
            "regressor": reg_name,
            **metrics,
        }
        combination_results.append(row)

        del reg
        gc.collect()

    del clf
    gc.collect()

classifier_eval_df = pd.DataFrame(classifier_eval_rows).sort_values("F1_Zero", ascending=False)
regressor_eval_df = pd.DataFrame(regressor_eval_rows).groupby("regressor").mean().reset_index()
combo_eval_df = pd.DataFrame(combination_results).sort_values("Total_Cost")

classifier_eval_df


In [ ]:
# Display regressor-only evaluation results (non-zero targets).
regressor_eval_df


In [ ]:
# Display combined two-stage evaluation across all model pairs.
combo_eval_df


In [ ]:
# Plot ROC curves for all classifiers to compare zero detection quality.
plt.figure(figsize=(8, 6))
for clf_name, preds in classifier_predictions.items():
    fpr, tpr, _ = roc_curve(y_test_zero, preds["proba"])
    plt.plot(fpr, tpr, label=f"{clf_name} (AUC={auc(fpr, tpr):.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Classifier ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Plot non-zero regression performance for each regressor.
nonzero_test_mask = y_test.values > 0
plt.figure(figsize=(8, 6))
np.random.seed(42)
sample_idx = np.random.choice(np.where(nonzero_test_mask)[0], size=min(2000, nonzero_test_mask.sum()), replace=False)
for reg_name, preds in regressor_predictions.items():
    plt.scatter(y_test.values[sample_idx], preds[sample_idx], s=8, alpha=0.3, label=reg_name)

max_val = max(y_test.values[nonzero_test_mask].max(), max(preds[nonzero_test_mask].max() for preds in regressor_predictions.values()))
plt.plot([0, max_val], [0, max_val], color="black", linewidth=1)
plt.xlabel("Actual Demand (Non-Zero)")
plt.ylabel("Predicted Demand")
plt.title("Regressor Predictions on Non-Zero Samples")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Visualize combined two-stage forecasts for top-demand products.
use_cols = ["stock_code", "date", "demand_qty"]
max_date = None
for chunk in pd.read_csv(tabular_path, usecols=use_cols, chunksize=200_000, low_memory=False):
    chunk["date"] = pd.to_datetime(chunk["date"])
    chunk_max = chunk["date"].max()
    max_date = chunk_max if max_date is None else max(max_date, chunk_max)

cutoff = max_date - pd.Timedelta(days=30)
test_chunks = []
for chunk in pd.read_csv(tabular_path, usecols=use_cols, chunksize=200_000, low_memory=False):
    chunk["date"] = pd.to_datetime(chunk["date"])
    test_chunk = chunk[chunk["date"] > cutoff]
    if not test_chunk.empty:
        test_chunks.append(test_chunk)

df_test = pd.concat(test_chunks, ignore_index=True)
df_test = df_test.sort_values(["stock_code", "date"]).reset_index(drop=True)

if len(df_test) != len(y_test):
    raise ValueError("Mismatch between test rows and predictions.")

df_test["y_true"] = y_test.values

top_k = 3
top_products = (
    df_test.groupby("stock_code")["y_true"]
    .sum()
    .sort_values(ascending=False)
    .head(top_k)
    .index
)

fig, axes = plt.subplots(top_k, 1, figsize=(12, 3 * top_k), sharex=True)
if top_k == 1:
    axes = [axes]

for ax, stock_code in zip(axes, top_products):
    subset = df_test[df_test["stock_code"] == stock_code]
    ax.plot(subset["date"], subset["y_true"], label="Actual", linewidth=1.5, color="black")
    for model_name, preds in combined_predictions.items():
        ax.plot(subset["date"], preds[subset.index], label=model_name, linewidth=1)
    ax.set_title(f"Stock {stock_code} - Combined Two-Stage Forecasts")
    ax.set_ylabel("Demand")
    ax.legend(ncol=2, fontsize=7)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()
